In [1]:
import pandas as pd

metadata = pd.read_csv("dataset/HAM10000_metadata.csv")
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [2]:
metadata['dx'].value_counts()

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [3]:
 import os

os.makedirs("Skin Cancer Binary/Benign", exist_ok=True)
os.makedirs("Skin Cancer Binary/Malignant", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [4]:
import os
import shutil

part1 = "dataset/HAM10000_images_part_1"
part2 = "dataset/HAM10000_images_part_2"

image_dict = {}

for img in os.listdir(part1):
    image_dict[img.split('.')[0]] = os.path.join(part1, img)

for img in os.listdir(part2):
    image_dict[img.split('.')[0]] = os.path.join(part2, img)

print("Total Images Found:", len(image_dict))

Total Images Found: 10015


In [5]:
mel_count = len(metadata[metadata['dx'] == 'mel'])
print("Malignant images:", mel_count)

Malignant images: 1113


In [6]:
import os
import shutil
import random

# Create folders
os.makedirs("Skin Cancer Binary/Benign", exist_ok=True)
os.makedirs("Skin Cancer Binary/Malignant", exist_ok=True)

# Classes
benign_classes = ['nv', 'bkl', 'bcc', 'akiec', 'vasc', 'df']

# All malignant images
malignant_df = metadata[metadata['dx'] == 'mel']

# All benign images
benign_df = metadata[metadata['dx'].isin(benign_classes)]

# Randomly select 1113 benign images
benign_sample = benign_df.sample(n=1113, random_state=42)

# Copy malignant images
for _, row in malignant_df.iterrows():
    image_id = row['image_id']
    source_path = image_dict.get(image_id)

    if source_path:
        shutil.copy(source_path, "Skin Cancer Binary/Malignant")

# Copy selected benign images
for _, row in benign_sample.iterrows():
    image_id = row['image_id']
    source_path = image_dict.get(image_id)

    if source_path:
        shutil.copy(source_path, "Skin Cancer Binary/Benign")

print("Balanced dataset created successfully!")

Balanced dataset created successfully!


In [7]:
print("Benign:", len(os.listdir("Skin Cancer Binary/Benign")))
print("Malignant:", len(os.listdir("Skin Cancer Binary/Malignant")))

Benign: 1113
Malignant: 1113


In [8]:
import os
import shutil
import random

source_dir = "Skin Cancer Binary"
split_dir = "Skin Cancer Binary Split"

# Create folders
for split in ["train", "val", "test"]:
    for cls in ["Benign", "Malignant"]:
        os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

for cls in ["Benign", "Malignant"]:

    files = os.listdir(os.path.join(source_dir, cls))
    random.shuffle(files)

    train_end = int(0.8 * len(files))
    val_end = int(0.9 * len(files))

    train_files = files[:train_end]
    val_files = files[train_end:val_end]
    test_files = files[val_end:]

    for file in train_files:
        shutil.copy(
            os.path.join(source_dir, cls, file),
            os.path.join(split_dir, "train", cls, file)
        )

    for file in val_files:
        shutil.copy(
            os.path.join(source_dir, cls, file),
            os.path.join(split_dir, "val", cls, file)
        )

    for file in test_files:
        shutil.copy(
            os.path.join(source_dir, cls, file),
            os.path.join(split_dir, "test", cls, file)
        )

print("Dataset split completed!")

Dataset split completed!


In [9]:
import os

print("Train Benign:", len(os.listdir("Skin Cancer Binary Split/train/Benign")))
print("Train Malignant:", len(os.listdir("Skin Cancer Binary Split/train/Malignant")))

print("Val Benign:", len(os.listdir("Skin Cancer Binary Split/val/Benign")))
print("Val Malignant:", len(os.listdir("Skin Cancer Binary Split/val/Malignant")))

print("Test Benign:", len(os.listdir("Skin Cancer Binary Split/test/Benign")))
print("Test Malignant:", len(os.listdir("Skin Cancer Binary Split/test/Malignant")))

Train Benign: 890
Train Malignant: 890
Val Benign: 111
Val Malignant: 111
Test Benign: 112
Test Malignant: 112


In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    "Skin Cancer Binary Split/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_test_datagen.flow_from_directory(
    "Skin Cancer Binary Split/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = val_test_datagen.flow_from_directory(
    "Skin Cancer Binary Split/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 1780 images belonging to 2 classes.
Found 222 images belonging to 2 classes.
Found 224 images belonging to 2 classes.
